<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/agent.png" align="center" width="20%">
</div>

<br>

# EVALUATING AGENT ROUTER AND SKILLS USING EDD

<br>

**About:** Run LLM-as-judge and programmatic evaluations on a traced agent's spans, score tool selection, SQL generation, code runnability, and response clarity, then write scores back to Arize Phoenix to close the EDD loop.

**Learning Goals:** (1) Query Phoenix spans using `SpanQuery` to retrieve evaluation inputs. (2) Apply tool-calling evaluations using Phoenix's built-in `TOOL_CALLING_PROMPT_TEMPLATE`. (3) Write custom LLM-as-judge prompts for domain-specific quality dimensions. (4) Implement a programmatic (non-LLM) eval for code runnability. (5) Write evaluation scores back to Phoenix and interpret results to identify improvement opportunities.

**Keywords:** llm-as-judge, phoenix evals, tool calling eval, span query, agent evaluation, edd

**Prerequisite Knowledge:** (1) `01_evaluating_agents.ipynb` - agent tools and router, (2) `02_tracing_agents.ipynb` - Phoenix setup, span types, and trace collection

**Target User:** ML engineers and data practitioners who want to systematically evaluate and improve multi-tool LLM agents beyond manual inspection.

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 0: SETUP](#Part_0)
> #### [PART 1: RUNNING THE AGENT FOR EVAL COLLECTION](#Part_1)
> #### [PART 2: TOOL CALLING EVALUATION](#Part_2)
> #### [PART 3: SKILL-LEVEL EVALUATIONS](#Part_3)
> #### [PART 4: EVALUATION-DRIVEN IMPROVEMENT](#Part_4)

<br>

<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SETUP** AND ENVIRONMENT

Notebook 02 gave the agent a memory - every LLM call, tool invocation, and intermediate step now lands in Phoenix as a structured span. Reading traces by hand works for a handful of queries. It does not scale to a hundred, and it cannot answer questions like "how often does the router pick the wrong tool?" or "what fraction of generated chart code actually runs?" without a spreadsheet and a stopwatch.

**This notebook closes the EDD loop by turning traces into scores.** Four evaluations run against the spans collected in notebook 02:

- **Tool calling** - did the router pick the right tool for the query? (LLM-as-judge, Phoenix built-in)
- **Code runnability** - does the generated chart code actually execute? (Programmatic, no LLM)
- **Response clarity** - is the final answer precise and coherent? (LLM-as-judge, custom prompt)
- **SQL generation** - does the generated SQL correctly answer the question? (LLM-as-judge, custom prompt)

Each score is written back to Phoenix as an annotation on the original span, so a rising or falling score is directly attributable to the span that produced it. That is the mechanic that makes "improve one thing at a time" tractable: after a prompt rewrite, you re-run the affected eval and read the delta.

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/edd_loop.png" align="center" width="55%" padding="10"><br>
    <br>
    This notebook implements the "Evaluate" and "Improve" stages of the cycle. Notebook 02 handled "Instrument" and "Collect".
</div>

**How this notebook is organized**

- **Part 1** re-declares the agent (identical to notebook 02) and runs it against a batch of six questions so Phoenix has span data to evaluate.
- **Part 2** runs the built-in Phoenix tool-calling evaluation against the router's LLM spans.
- **Part 3** runs the three skill-level evaluations (code runnability, response clarity, SQL generation).
- **Part 4** applies the improved SQL prompt from notebooks 01 and 02 as the demonstration fix, closing the loop end to end.

The environment requires an OpenAI API key in `.env` as `OPENAI_API_KEY`, Phoenix running locally, and the same Parquet dataset used in the earlier notebooks. All imports go in the first code cell.

___

**Note:** `nest_asyncio.apply()` is required because Phoenix's evaluation functions use `asyncio` internally, and Jupyter's own event loop would otherwise block them. If you run these evals in a standalone Python script, `nest_asyncio` is unnecessary.

___

In [ ]:
from openai import OpenAI
import pandas as pd
import os
import json
import duckdb
import pydantic
from pydantic import BaseModel, Field
from IPython.display import Markdown
from typing import Any, Dict, List, Sequence, Union
from dotenv import load_dotenv

# Phoenix + OpenInference tracing stack.
# Verified against arize-phoenix + arize-phoenix-otel + openinference-instrumentation-openai
# packages available on PyPI as of 2026-08-31. Re-check import paths at
# https://arize.com/docs/phoenix/tracing/how-to-tracing/setup-tracing/instrument
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.trace import StatusCode
from openinference.instrumentation import suppress_tracing

from tqdm import tqdm
from phoenix.evals import (
    TOOL_CALLING_PROMPT_TEMPLATE,
    llm_classify,
    OpenAIModel
)
from phoenix.trace import SpanEvaluations
from phoenix.trace.dsl import SpanQuery

import nest_asyncio
nest_asyncio.apply()

load_dotenv()
AI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=AI_API_KEY)

# The agent under test runs on the same model as notebooks 01 and 02.
# The LLM-as-judge model (used inside llm_classify calls below) is intentionally
# stronger (gpt-4o). Using a stronger judge than the system under test is
# standard practice - it reduces the chance the judge shares the same blind spots
# as the model whose output it is grading.
AGENT_MODEL = "gpt-4o-mini"
JUDGE_MODEL = "gpt-4o"
MODEL = AGENT_MODEL  # legacy alias used by the agent code below
TRANSACTION_DATA_FILE_PATH = "data/Store_Sales_Price_Elasticity_Promotions_Data.parquet"

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **This notebook imports `suppress_tracing` from `openinference.instrumentation`. Write one sentence explaining when you would use it, and why running an LLM-as-judge evaluation without suppressing tracing would pollute your Phoenix project.**

<br>

```python
# Write your explanation as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **RUNNING THE AGENT FOR EVAL COLLECTION**: Generating Span Data

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/spans_agent.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

The agent and Phoenix setup are reproduced from notebook 02. The agent must run against a batch of questions to generate enough spans for evaluation.

The six test questions below were chosen to cover different tool paths:

- Questions 1-2 trigger `lookup_sales_data` only (no visualization, no analysis)
- Questions 3-4 trigger `lookup_sales_data` and `analyze_sales_data`
- Questions 5-6 trigger `lookup_sales_data` and `generate_visualization`

A well-formed batch exercises every tool and every tool combination so that evals have data on each path. A batch that only triggers one tool cannot evaluate the others.

<strong style="color:red">KEY CONSIDERATION:</strong> The `DependencyConflict` warning about `openai >= 1.69.0` that appears when registering the tracer provider is a Phoenix version mismatch with the installed OpenAI SDK. It does not prevent tracing from working, but the warning is a signal to pin library versions in `requirements.txt` before deploying to production.

In [ ]:
PROJECT_NAME = "evaluating_agent_router_and_skills"
PHOENIX_COLLECTOR_ENDPOINT = "http://localhost:6006/v1/traces"

tracer_provider = register(
    project_name=PROJECT_NAME,
    endpoint=PHOENIX_COLLECTOR_ENDPOINT,
    auto_instrument=True
)

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = tracer_provider.get_tracer(__name__)

In [ ]:
SQL_GENERATION_PROMPT = """
Generate an SQL query based on the prompt that follows. Do not reply with anything besides the SQL query.
The prompt is: {prompt}

The available columns are: {columns}
The table name is: {table_name}
"""

DATA_ANALYSIS_PROMPT = """
Analyze the following data: {data}
Your job is to answer the following question: {prompt}
"""

CHART_CONFIGURATION_PROMPT = """
Generate a chart configuration based on this data: {data}
The goal is to show: {visualization_goal}
"""

CREATE_CHART_PROMPT = """
Write Python code to create a chart based on the configuration below.
Return only the code, no other text.
config: {config}
"""


class VisualizationConfig(BaseModel):
    """Configuration schema for chart generation."""
    chart_type: str = Field(..., description="Type of chart (e.g., bar, line, scatter).")
    x_axis: str = Field(..., description="Column name for the x-axis.")
    y_axis: str = Field(..., description="Column name for the y-axis.")
    title: str = Field(..., description="Chart title.")


def generate_sql_query(prompt: str, columns: list[str], table_name: str) -> str:
    """Generate SQL from a natural language prompt (auto-instrumented by OpenAIInstrumentor)."""
    formatted_prompt = SQL_GENERATION_PROMPT.format(
        prompt=prompt, columns=columns, table_name=table_name
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    return response.choices[0].message.content.strip()


@tracer.tool()
def lookup_sales_data(prompt: str) -> str:
    """Query sales data using LLM-generated SQL. Emits a tool-typed span."""
    try:
        table_name = "sales"
        df = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df")
        sql_query = generate_sql_query(prompt, df.columns.tolist(), table_name)
        sql_query = sql_query.strip().replace("```sql", "").replace("```", "")
        result = duckdb.sql(sql_query).df()
        return result.to_string()
    except Exception as e:
        return f"Error accessing data: {e}"


@tracer.tool()
def analyze_sales_data(prompt: str, data: str) -> str:
    """Analyze sales data with an LLM. Emits a tool-typed span."""
    formatted_prompt = DATA_ANALYSIS_PROMPT.format(data=data, prompt=prompt)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    analysis = response.choices[0].message.content
    return analysis if analysis else "No analysis could be generated"


@tracer.chain()
def extract_chart_config(data: str, visualization_goal: str) -> dict:
    """Generate a chart configuration. Emits a chain-typed span."""
    formatted_prompt = CHART_CONFIGURATION_PROMPT.format(
        data=data, visualization_goal=visualization_goal
    )
    # Structured Outputs. Verified against openai>=1.50 SDK, 2026-08-31.
    # See notebook 01 Part 1.3 for the API-version note.
    response = client.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
        response_format=VisualizationConfig,
    )
    try:
        content = response.choices[0].message.parsed
        return {"chart_type": content.chart_type, "x_axis": content.x_axis,
                "y_axis": content.y_axis, "title": content.title, "data": data}
    except Exception:
        return {"chart_type": "bar", "x_axis": "date", "y_axis": "value",
                "title": visualization_goal, "data": data}


@tracer.chain()
def create_chart(config: dict) -> str:
    """Generate chart code. Emits a chain-typed span."""
    formatted_prompt = CREATE_CHART_PROMPT.format(config=config)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    code = response.choices[0].message.content
    code = code.replace("```python", "").replace("```", "").strip()
    return code


@tracer.chain()
def generate_visualization(data: str, visualization_goal: str) -> str:
    """Generate visualization code. Emits a chain-typed span."""
    config = extract_chart_config(data, visualization_goal)
    code = create_chart(config)
    return code

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_sales_data",
            "description": "Look up sales transaction data from the Store Sales Price Elasticity Promotions dataset",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_sales_data",
            "description": "Analyze sales data to extract business insights",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool output."},
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["data", "prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_visualization",
            "description": "Generate executable Python code to create a data visualization",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool output."},
                    "visualization_goal": {"type": "string", "description": "Description of the chart to create."}
                },
                "required": ["data", "visualization_goal"]
            }
        }
    }
]

tool_implementations = {
    "lookup_sales_data": lookup_sales_data,
    "analyze_sales_data": analyze_sales_data,
    "generate_visualization": generate_visualization
}

SYSTEM_PROMPT = """
You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.
"""


@tracer.chain()
def handle_tool_calls(tool_calls, messages):
    """Execute tool calls and append results. Emits a chain-typed span."""
    for tool_call in tool_calls:
        function = tool_implementations[tool_call.function.name]
        function_args = json.loads(tool_call.function.arguments)
        result = function(**function_args)
        messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    return messages


def run_agent(messages, system_prompt: str = SYSTEM_PROMPT):
    """Run the router loop until the LLM produces a final answer."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not any(isinstance(m, dict) and m.get("role") == "system" for m in messages):
        messages.insert(0, {"role": "system", "content": system_prompt})
    while True:
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
        )
        messages.append(response.choices[0].message)
        tool_calls = response.choices[0].message.tool_calls
        if tool_calls:
            messages = handle_tool_calls(tool_calls, messages)
        else:
            return response.choices[0].message.content


def start_main_span(messages):
    """Wrap the entire agent run in a root agent-typed span."""
    with tracer.start_as_current_span("AgentRun", openinference_span_kind="agent") as span:
        span.set_input(value=messages)
        result = run_agent(messages)
        span.set_output(value=result)
        span.set_status(StatusCode.OK)
        return result

In [ ]:
# Run the agent on a diverse batch of questions to generate spans for evaluation.
# This batch is designed to exercise all three tools and multiple tool combinations.
agent_questions = [
    "What was the most popular product SKU?",
    "What was the total revenue across all stores?",
    "Which store had the highest sales volume?",
    "Create a bar chart showing total sales by store",
    "What percentage of items were sold on promotion?",
    "What was the average transaction value?"
]

for question in tqdm(agent_questions, desc="Collecting traces"):
    try:
        start_main_span([{"role": "user", "content": question}])
    except Exception as e:
        print(f"Error on question: {question!r} - {e}")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The batch above uses 6 questions. Why is a small batch like this sufficient for validating that evaluations are working, but insufficient for making confident claims about overall agent quality?**

<br>

```python
# Write your reasoning as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **TOOL CALLING EVALUATION**: Did the Router Choose the Right Tool?

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/tool2.png" align="center" width="20%" padding="10"><br>
    <br>
</div>

Tool calling evaluation answers the question: "Given this user query, did the router call the right tool with the right parameters?"

Before diving into the specific mechanics of each evaluation, it helps to have the whole suite in view. The four evals in this notebook are not interchangeable - each targets a different span kind, uses a different judgement mechanism, and surfaces a different class of failure.

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/eval_matrix.png" align="center" width="80%" padding="10"><br>
    <br>
    The four evaluations covered in this notebook, mapped to the spans they query and the failures they surface.
</div>

**Back to tool calling.** Phoenix provides a built-in `TOOL_CALLING_PROMPT_TEMPLATE` that formats this as a binary judgment: `correct` or `incorrect`. The judge LLM receives the user's question, the tool call the router made, and the tool definitions, then decides whether the tool call was appropriate.

**How it works:**

1. Query Phoenix for LLM-typed spans that have tool calls (`llm.tools` is not null).
2. Format each span's input (the question) and output (the tool call) into the evaluation template.
3. Pass each formatted row to `llm_classify`, which calls the judge LLM for each row.
4. Write the results back to Phoenix so they appear alongside the original spans in the UI.

___

**Note:** `suppress_tracing()` wraps the evaluation calls. Without it, the judge LLM calls would generate new LLM spans in the Phoenix project, mixing eval traffic with agent traffic and making traces uninterpretable.

___

In [ ]:
# Step 1: Query LLM spans that have tool calls.
query = SpanQuery().where(
    "span_kind == 'LLM'",
).select(
    question="input.value",
    tool_call="llm.tools"
)

tool_calls_df = px.Client().query_spans(
    query,
    project_name=PROJECT_NAME,
    timeout=None
)

# Drop rows where the LLM made no tool call (final-answer turns).
tool_calls_df = tool_calls_df.dropna(subset=["tool_call"])
print(f"Spans with tool calls: {len(tool_calls_df)}")
tool_calls_df.head()

In [ ]:
# Step 2: Run the tool calling evaluation using Phoenix's built-in template.
# suppress_tracing() prevents the judge LLM calls from appearing as new spans.
with suppress_tracing():
    tool_call_eval = llm_classify(
        data=tool_calls_df,
        template=TOOL_CALLING_PROMPT_TEMPLATE.template[0].template.replace(
            "{tool_definitions}",
            json.dumps(tools).replace("{", '"').replace("}", '"')
        ),
        rails=["correct", "incorrect"],
        model=OpenAIModel(model=JUDGE_MODEL),
        provide_explanation=True
    )

tool_call_eval["score"] = tool_call_eval.apply(
    lambda x: 1 if x["label"] == "correct" else 0, axis=1
)
tool_call_eval.head()

In [ ]:
# Step 3: Write evaluation scores back to Phoenix.
from phoenix.client import Client

Client().spans.log_span_annotations_dataframe(
    dataframe=tool_call_eval,
    annotation_name="ToolCallingEval",
    annotator_kind="LLM",
    sync=True
)

print(f"Tool calling eval complete. Mean score: {tool_call_eval['score'].mean():.2f}")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The tool calling template uses `replace('{', '"').replace('}', '"')` to serialize the tool definitions. This is a workaround for a string formatting collision. Explain what the collision is and why a more robust approach would use `json.dumps` with a custom escaping strategy or a different template format.**

<br>

```python
# Write your explanation as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **SKILL-LEVEL EVALUATIONS**: Code, Clarity, and SQL Quality

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/tool3.png" align="center" width="20%" padding="10"><br>
    <br>
</div>

#### CONTENTS:

> [PART 3.1: CODE RUNNABILITY EVAL](#Part_3_1)<br>
> [PART 3.2: RESPONSE CLARITY EVAL](#Part_3_2)<br>
> [PART 3.3: SQL GENERATION EVAL](#Part_3_3)<br>

<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: CODE RUNNABILITY EVAL

<br>

The visualization tool generates Python code as a string. Whether the code actually runs is not something an LLM judge can reliably verify - it requires executing the code. This makes code runnability a natural programmatic eval: no LLM, no API cost, deterministic result.

The `code_is_runnable` function calls `exec()` on the generated code and returns `True` if it runs without exception. This is a fast, cheap, binary signal. It does not check whether the chart looks correct - only whether the code executes.

___

**Note:** Running `exec()` on LLM-generated code is inherently unsafe in production. For this evaluation in a controlled notebook environment, it is acceptable. In a production setting, you would run the generated code in a sandboxed subprocess.

In [ ]:
# Query spans for the generate_visualization chain to get generated code.
query = SpanQuery().where(
    "name == 'generate_visualization'"
).select(
    generated_code="output.value"
)

code_gen_df = px.Client().query_spans(
    query,
    project_name=PROJECT_NAME,
    timeout=None
)
print(f"Visualization spans: {len(code_gen_df)}")
code_gen_df.head()

In [ ]:
def code_is_runnable(output: str) -> bool:
    """
    Test whether generated Python code executes without raising an exception.

    Parameters
    ----------
    output : str
        Python code string, possibly wrapped in markdown code fences.

    Returns
    -------
    bool
        True if the code runs to completion, False if any exception is raised.

    Note
    ----
    Uses exec() in the current process. Do not use in production environments
    without sandboxing - exec() can run arbitrary code.
    """
    output = output.strip().replace("```python", "").replace("```", "")
    try:
        exec(output)
        return True
    except Exception:
        return False


code_gen_df["label"] = code_gen_df["generated_code"].apply(code_is_runnable).map(
    {True: "runnable", False: "not_runnable"}
)
code_gen_df["score"] = code_gen_df["label"].map({"runnable": 1, "not_runnable": 0})
print(code_gen_df[["label", "score"]])

In [ ]:
# Write code runnability scores back to Phoenix.
px.Client().log_evaluations(
    SpanEvaluations(eval_name="Runnable Code Eval", dataframe=code_gen_df),
)
print(f"Code runnability eval complete. Mean score: {code_gen_df['score'].mean():.2f}")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The `code_is_runnable` function strips markdown fences before calling `exec()`. Write a one-sentence explanation of why the generated code might contain markdown fences, and name the specific line in `create_chart` that attempts to handle this upstream.**

<br>

```python
# Write your explanation and point to the specific line as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: RESPONSE CLARITY EVAL

<br>

Response clarity is a judgment that cannot be computed programmatically - it requires understanding. This is where LLM-as-judge earns its place.

The clarity judge receives the user's original question and the agent's final answer, then classifies the response as `clear` or `unclear`. The custom prompt below is written to guide the judge toward actionable distinctions:

- A `clear` response is precise, coherent, and directly addresses the query.
- An `unclear` response is vague, disorganized, or requires inference to interpret.

The prompt asks for an explanation before the label. This serves two purposes: it forces the judge to reason before committing (reducing label noise), and it gives you a readable explanation for why a specific response was marked unclear, which is directly actionable.

In [ ]:
CLARITY_LLM_JUDGE_PROMPT = """
In this task, you will be presented with a query and an answer. Your objective is to evaluate the clarity
of the answer in addressing the query. A clear response is one that is precise, coherent, and directly
addresses the query without introducing unnecessary complexity or ambiguity. An unclear response is one
that is vague, disorganized, or difficult to understand, even if it may be factually correct.

Your response should be a single word: either "clear" or "unclear," and it should not include any other
text or characters. "clear" indicates that the answer is well-structured, easy to understand, and
appropriately addresses the query. "unclear" indicates that some part of the response could be better
structured or worded.
Please carefully consider the query and answer before determining your response.

After analyzing the query and the answer, you must write a detailed explanation of your reasoning to
justify why you chose either "clear" or "unclear." Avoid stating the final label at the beginning of your
explanation. Your reasoning should include specific points about how the answer does or does not meet the
criteria for clarity.

[BEGIN DATA]
Query: {query}
Answer: {response}
[END DATA]
Please analyze the data carefully and provide an explanation followed by your response.

EXPLANATION: Provide your reasoning step by step, evaluating the clarity of the answer based on the query.
LABEL: "clear" or "unclear"
"""

In [ ]:
# Query agent-typed spans for final responses.
query = SpanQuery().where(
    "span_kind == 'AGENT'"
).select(
    response="output.value",
    query="input.value"
)

clarity_df = px.Client().query_spans(
    query,
    project_name=PROJECT_NAME,
    timeout=None
)
print(f"Agent spans: {len(clarity_df)}")
clarity_df.head()

In [ ]:
# suppress_tracing() prevents judge LLM calls from appearing as new spans.
# Note: llm_classify takes its input dataframe as `data=`, matching the
# signature used in Part 2. Older tutorials used `dataframe=`, which no
# longer resolves in current phoenix-evals versions.
with suppress_tracing():
    clarity_eval = llm_classify(
        data=clarity_df,
        template=CLARITY_LLM_JUDGE_PROMPT,
        rails=["clear", "unclear"],
        model=OpenAIModel(model=JUDGE_MODEL),
        provide_explanation=True
    )

clarity_eval["score"] = clarity_eval.apply(
    lambda x: 1 if x["label"] == "clear" else 0, axis=1
)
print(f"Clarity eval complete. Mean score: {clarity_eval['score'].mean():.2f}")
clarity_eval.head()

In [ ]:
px.Client().log_evaluations(
    SpanEvaluations(eval_name="Response Clarity", dataframe=clarity_eval),
)
print("Clarity scores written to Phoenix.")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The clarity prompt asks the judge to explain its reasoning before giving the label. Name one concrete way this explanation can guide an improvement to the agent's system prompt, and write a revised SYSTEM_PROMPT line that addresses a hypothetical 'unclear' response pattern.**

<br>

```python
# Write your explanation and revised system prompt as comments.
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.3: SQL GENERATION EVAL

<br>

SQL generation quality is evaluated with a custom LLM judge that checks whether the generated SQL query correctly answers the user's intent. This is harder than the tool calling eval (which checks whether the right tool was called) - it checks whether the tool's internal LLM call produced a correct artifact.

The judge receives the user's instruction and the generated SQL query, then returns `correct` or `incorrect`. Importantly, the judge is instructed to assume the table and columns exist - it evaluates logical correctness, not syntax checking.

In [ ]:
SQL_EVAL_GEN_PROMPT = """
SQL Evaluation Prompt:
-----------------------
You are tasked with determining if the SQL generated appropriately answers a given instruction
taking into account its generated query and response.

Data:
-----
- [Instruction]: {question}
  This section contains the specific task or problem that the SQL query is intended to solve.

- [Reference Query]: {sql_gen}
  This is the SQL query submitted for evaluation. Analyze it in the context of the provided
  instruction.

Evaluation:
-----------
Your response should be a single word: either "correct" or "incorrect".
You must assume that the db exists and that columns are appropriately named.

- "correct" indicates that the SQL query correctly solves the instruction.
- "incorrect" indicates that the SQL query does not solve the instruction correctly.

Note: Your response should contain only the word "correct" or "incorrect" with no additional text
or characters.
"""

In [ ]:
# Query LLM spans to get SQL generation inputs and outputs.
query = SpanQuery().where(
    "span_kind == 'LLM'"
).select(
    sql_gen="output.value",
    question="input.value",
)

sql_df = px.Client().query_spans(
    query,
    project_name=PROJECT_NAME,
    timeout=None
)
print(f"LLM spans: {len(sql_df)}")
sql_df.head()

In [ ]:
with suppress_tracing():
    sql_gen_eval = llm_classify(
        data=sql_df,
        template=SQL_EVAL_GEN_PROMPT,
        rails=["correct", "incorrect"],
        model=OpenAIModel(model=JUDGE_MODEL),
        provide_explanation=True
    )

sql_gen_eval["score"] = sql_gen_eval.apply(
    lambda x: 1 if x["label"] == "correct" else 0, axis=1
)
print(f"SQL eval complete. Mean score: {sql_gen_eval['score'].mean():.2f}")
sql_gen_eval.head()

In [ ]:
px.Client().log_evaluations(
    SpanEvaluations(eval_name="SQL Gen Eval", dataframe=sql_gen_eval),
)
print("SQL gen scores written to Phoenix.")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The SQL gen eval above queries all LLM spans, not just the ones that generated SQL. That means it also runs the SQL judge prompt over the analysis and chart-config LLM spans, which will score them as `incorrect` for the wrong reason. Write a `SpanQuery` that filters to only the SQL generation LLM spans.**

<br>

The Phoenix `SpanQuery.where()` clause takes a string expression that can reference span attributes. Two common patterns for narrowing LLM spans:

- **By parent span name**: `"parent.name == 'lookup_sales_data'"` selects LLM spans that fired inside the `lookup_sales_data` tool.
- **By input text substring**: `"'Generate an SQL query' in input.value"` selects LLM spans whose prompt contains a signature string from `SQL_GENERATION_PROMPT`.

Either works. The first is more robust (no dependence on prompt wording); the second is more literal (no dependence on span parentage).

```python
# Write the corrected SpanQuery below. Try both approaches and compare row counts.
query = SpanQuery().where(
    ...
).select(
    sql_gen='output.value',
    question='input.value',
)
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **EVALUATION-DRIVEN IMPROVEMENT**: Reading Results and Closing the Loop

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/agent_dark_background.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

With all three eval scores written to Phoenix, the EDD loop is visible in one place:

- **Tool calling score** < 1.0 - the router is selecting the wrong tool for some queries. Fix: rewrite the tool description that the failing queries trigger.
- **Code runnability score** < 1.0 - the visualization tool's code generation is producing non-runnable code. Fix: strengthen the `CREATE_CHART_PROMPT` with explicit format constraints or add a retry on failure.
- **SQL gen score** < 1.0 - the SQL generation is producing incorrect queries for some questions. Fix: use the improved SQL prompt from notebooks 01 and 02.
- **Clarity score** < 1.0 - the agent's final responses are unclear for some questions. Fix: revise `SYSTEM_PROMPT` to specify the expected response format.

The improved SQL prompt is applied below, closing the loop for the SQL generation failure mode. After applying any change, re-run the agent batch and re-run the affected eval to confirm the score improves.

___

**Note:** Improving one eval score sometimes hurts another. For example, a more constrained SQL prompt might return `ERROR: Ambiguous or Unsupported Query` for a valid-but-ambiguous question, which would reduce the SQL gen score but the router would then correctly fail to call the tool. Always re-run all evals after a change, not just the one you targeted.

___

In [ ]:
# Improved SQL prompt - targets the column hallucination failure mode identified in notebooks 01-02.
# Note: leading whitespace inside a triple-quoted string is sent to the model as-is,
# so this string is deliberately flush left.
SQL_GENERATION_PROMPT_IMPROVED = """You are an expert SQL developer. Your task is to generate a syntactically valid SQL query
that answers the user's request based on the provided table schema.
You must return **only** the SQL code, formatted cleanly with uppercase SQL keywords.

Follow these steps internally before producing the final answer:
1. Analyze the user's intent.
2. Identify which columns and filters are relevant.
3. Construct a valid SQL SELECT statement that answers the query.

Schema Information:
- Table Name: {table_name}
- Available Columns: {columns}

User Request:
"{prompt}"

Guidelines:
- Return only the SQL query (no explanations, comments, or markdown).
- Use exact column names as provided - do not invent or rename any.
- Prefer simple, interpretable SQL (avoid unnecessary nesting or joins).
- If the user request is ambiguous or impossible, return:
  SELECT 'ERROR: Ambiguous or Unsupported Query' AS message;
"""


def generate_sql_query_v2(prompt: str, columns: list[str], table_name: str) -> str:
    """SQL generation using the improved prompt."""
    formatted_prompt = SQL_GENERATION_PROMPT_IMPROVED.format(
        prompt=prompt, columns=columns, table_name=table_name
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": formatted_prompt}],
    )
    return response.choices[0].message.content.strip()


# Rebuild lookup_sales_data with the improved SQL prompt.
@tracer.tool()
def lookup_sales_data_v2(prompt: str) -> str:
    """Query sales data using the improved SQL generation prompt."""
    try:
        table_name = "sales"
        df = pd.read_parquet(TRANSACTION_DATA_FILE_PATH)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df")
        sql_query = generate_sql_query_v2(prompt, df.columns.tolist(), table_name)
        sql_query = sql_query.strip().replace("```sql", "").replace("```", "")
        result = duckdb.sql(sql_query).df()
        return result.to_string()
    except Exception as e:
        return f"Error accessing data: {e}"

In [ ]:
# Swap the improved lookup tool into the router's dispatch table, then re-run
# a short comparison batch through the fully traced agent. This produces a new
# set of SQL generation spans that we can score against the SQL gen eval and
# compare to the baseline score printed earlier.
#
# Two things to notice:
#   1. Only one thing changed - the SQL prompt inside lookup_sales_data. Every
#      other tool, prompt, and description is identical to the baseline run.
#      That is the "one change per cycle" discipline in action.
#   2. The comparison batch is intentionally small (3 queries) to keep the
#      re-run cheap. Larger batches give tighter score estimates; small batches
#      are enough to detect an obvious improvement or regression.

# Capture the baseline score before overwriting anything.
baseline_sql_score = sql_gen_eval["score"].mean()

# Point the router at the improved tool for the comparison batch.
tool_implementations["lookup_sales_data"] = lookup_sales_data_v2

comparison_questions = [
    # The classic column-hallucination trap: no "discount_percentage" column exists.
    "Show me all sales where the discount_percentage is above 20 percent.",
    # A well-formed lookup that the baseline handles fine (regression check).
    "What was the total revenue across all stores?",
    # A lookup that requires an aggregation the baseline sometimes gets wrong.
    "Which store had the highest total sales?",
]

for q in tqdm(comparison_questions, desc="Re-running with improved prompt"):
    try:
        start_main_span([{"role": "user", "content": q}])
    except Exception as e:
        print(f"Error on comparison question {q!r}: {e}")

# Re-query the SQL generation LLM spans (now contains both baseline and v2 spans).
# In a real project you would tag the project or the batch to keep them separate;
# here we simply re-run the same eval over the enlarged span set.
sql_df_v2 = px.Client().query_spans(
    SpanQuery().where("span_kind == 'LLM'").select(
        sql_gen="output.value", question="input.value"
    ),
    project_name=PROJECT_NAME,
    timeout=None,
)

with suppress_tracing():
    sql_gen_eval_v2 = llm_classify(
        data=sql_df_v2,
        template=SQL_EVAL_GEN_PROMPT,
        rails=["correct", "incorrect"],
        model=OpenAIModel(model=JUDGE_MODEL),
        provide_explanation=True,
    )
sql_gen_eval_v2["score"] = sql_gen_eval_v2["label"].eq("correct").astype(int)
improved_sql_score = sql_gen_eval_v2["score"].mean()

print(f"Baseline SQL gen mean score: {baseline_sql_score:.2f}")
print(f"Improved SQL gen mean score: {improved_sql_score:.2f}")
print(f"Delta: {improved_sql_score - baseline_sql_score:+.2f}")

# Restore the baseline tool so subsequent notebook cells (if any) see the
# original state - hygiene for readers who re-run cells out of order.
tool_implementations["lookup_sales_data"] = lookup_sales_data

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **You have run evals and found that `sql_gen_eval` shows a mean score of 0.7 (70% of SQL calls judged correct). Write a plan for one targeted change and describe how you would confirm it worked - what would you re-run, and what score change would count as a success?**

<br>

```python
# Write your improvement plan as comments:
# Change: ...
# Re-run: ...
# Success criterion: ...
```

<hr style="border: 2px solid#003262;" />

___

**Series complete.** Across three notebooks you have:

1. Built a three-tool LLM agent with a router loop (notebook 01).
2. Instrumented every LLM call, tool call, and pipeline step as OpenTelemetry spans in Arize Phoenix (notebook 02).
3. Scored the agent's spans with four evaluations - tool calling, code runnability, response clarity, SQL generation - and written the scores back to Phoenix as annotations.
4. Applied a targeted prompt improvement in response to an observed failure mode, re-run the affected batch, and read a measurable score delta.

That is the EDD cycle in full: instrument, collect, evaluate, improve. Repeat the loop for each failure mode you find. Change one thing at a time. Trust the scores over your intuition when they disagree - your intuition remembers the last five runs; the scores summarize the last five hundred.

**Where to go from here**

- Add more evaluations for failure modes that matter in your domain. Latency-per-tool, cost-per-query, and hallucination rate are common starting points.
- Grow the evaluation batch. Six queries validate that the eval pipeline works; a hundred queries validate that a prompt change actually moved the needle.
- Version prompts explicitly. The `_v2` suffix used here is a placeholder; a production workflow tracks prompts alongside eval scores in a lightweight prompt registry.

___

<hr style="border: 6px solid#003262;" />